<a href="https://colab.research.google.com/github/polreig/StartUp_DecoAI/blob/main/DecoAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalación de Dependencias
En esta celda instalamos el software necesario para que el ecosistema de nuestra startup funcione correctamente:
* **`google-genai`**: El SDK oficial para conectar con Gemini (nuestro motor de análisis visual y recomendación de compras).
* **`diffusers`, `transformers`, `accelerate`**: El conjunto de herramientas de Hugging Face para ejecutar la generación de imágenes mediante Inteligencia Artificial.
* **`opencv-python`**: La librería de visión artificial clásica que usaremos para extraer el plano/boceto de la habitación original.

In [ ]:
!pip install -q -U google-genai diffusers transformers accelerate opencv-python

## Conexión de APIs y Carga en GPU
Aquí preparamos el entorno de ejecución:
1. **Autenticación**: Cargamos de forma segura nuestra API Key para conectar con los servidores de Google (Gemini 2.5 Flash).
2. **Aceleración por Hardware (GPU)**: Descargamos los modelos pesados (`Stable Diffusion v1.5` y `ControlNet`) y los transferimos a la tarjeta gráfica (GPU) (`.to("cuda")`). Esto es vital para reducir el tiempo de renderizado de las imágenes de varios minutos a apenas unos segundos.

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image
import requests
from io import BytesIO
from google import genai
from google.colab import userdata
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel

print("1. Cargando credenciales de Gemini...")
GOOGLE_API_KEY = userdata.get('clave_API_gemini')
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

print("2. Cargando ControlNet y Stable Diffusion a la GPU...")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")

print("¡Sistemas listos!")

## Funciones principales de la Startup
Este bloque contiene el núcleo tecnológico de nuestro proyecto, dividido en dos funciones clave:

* **`transformar_habitacion()`**: Analiza la imagen original con un modelo de Visión-Lenguaje (VLM), extrae una lista de la compra dinámica con enlaces a tiendas reales (IKEA, Amazon, Leroy Merlin...), extrae la geometría de la habitación (Canny Edge) y genera las nuevas imágenes respetando la arquitectura original.
* **`obtener_3_estilos()`**: Actúa como el "Director Creativo". Analiza la foto y la petición del usuario para proponer 3 vías de diseño viables y personalizadas antes de empezar a generar imágenes.

In [ ]:
def transformar_habitacion(ruta_o_url, peticion_usuario):
    print("📥 Cargando imagen original...")

    if ruta_o_url.startswith('http'):
        response = requests.get(ruta_o_url)
        imagen_original = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        imagen_original = Image.open(ruta_o_url).convert("RGB")

    imagen_original = imagen_original.resize((512, 512))

    # Análisis con Gemini
    print("🧠 Analizando la habitación con Gemini...")
    prompt_gemini = f"""
    Eres un diseñador de interiores. El cliente dice: '{peticion_usuario}'.
    Analiza la foto y responde estrictamente con este formato:

    ANÁLISIS PARA EL CLIENTE:
    - Estilo Actual: [Tu análisis]
    - Recomendación: [Tu recomendación basada en lo que pide]
    - Paleta de Colores: [Colores]

    LISTA_DE_COMPRA:
    [Escribe una lista EXHAUSTIVA de TODOS los muebles, iluminación, textiles y objetos decorativos principales que componen esta habitación (mínimo entre 8 y 15 productos). Sigue ESTE FORMATO EXACTO por línea:]
    - Nombre del producto | Tienda recomendada (Elige una: IKEA, Leroy Merlin, Zara Home, Amazon) | Precio estimado en euros

    PROMPT_IMAGEN:
    [Escribe aquí UNA SOLA FRASE EN INGLÉS, separada por comas, describiendo la habitación recomendada.
    Ejemplo: "A cozy rustic living room, wooden furniture, warm lighting, highly detailed, 8k resolution, photorealistic, interior design"]
    """

    respuesta_gemini = gemini_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[imagen_original, prompt_gemini]
    )

    texto_completo = respuesta_gemini.text

    partes_prompt = texto_completo.split("PROMPT_IMAGEN:")
    texto_previo = partes_prompt[0]
    prompt_sd = partes_prompt[1].strip() if len(partes_prompt) > 1 else "modern interior design, photorealistic, 8k"

    partes_lista = texto_previo.split("LISTA_DE_COMPRA:")
    analisis_cliente = partes_lista[0].strip()
    texto_lista = partes_lista[1].strip() if len(partes_lista) > 1 else ""

    # Diseño visual y tienda
    html_links = """
    <div style='font-family: "Segoe UI", sans-serif; background: #ffffff; padding: 30px; border-radius: 12px; box-shadow: 0 8px 24px rgba(0,0,0,0.08); border: 1px solid #eef0f2; max-width: 1000px; margin: 20px auto;'>
        <h2 style='color: #1a1a1a; margin-top: 0; margin-bottom: 5px; font-size: 24px;'>🛍️ Catálogo de Opciones</h2>
        <p style='color: #666; font-size: 14px; margin-bottom: 20px;'>Compara precios en diferentes tiendas para los elementos clave de tu nuevo diseño.</p>
        <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); gap: 15px;'>
    """

    for linea in texto_lista.split('\n'):
        if '|' in linea and linea.strip().startswith('-'):
            datos = linea.replace('-', '', 1).strip().split('|')
            if len(datos) >= 3:
                nombre = datos[0].strip()
                tienda = datos[1].strip()
                precio = datos[2].strip()

                # Generador de enlaces para todas las tiendas
                query = urllib.parse.quote(nombre)
                l_ikea = f"https://www.ikea.com/es/es/search/?q={query}"
                l_amazon = f"https://www.amazon.es/s?k={query}"
                l_leroy = f"https://www.leroymerlin.es/buscar?q={query}"
                l_zara = f"https://www.zarahome.com/es/search.html?keyword={query}"

                html_links += f"""
                <div style='background: #f8f9fa; border: 1px solid #e9ecef; border-radius: 10px; padding: 16px;'>
                    <div style='display: flex; justify-content: space-between; align-items: start; margin-bottom: 12px;'>
                        <h4 style='margin: 0; color: #2d3436; font-size: 15px; font-weight: 600;'>{nombre.title()}</h4>
                        <span style='background: #e8f5e9; color: #2e7d32; padding: 4px 8px; border-radius: 6px; font-size: 13px; font-weight: 600; white-space: nowrap; margin-left: 10px;'>{precio}</span>
                    </div>
                    <p style='margin: 0 0 12px 0; font-size: 12px; color: #636e72;'>✨ Sugerencia IA: <b>{tienda}</b></p>
                    <p style='margin: 0 0 8px 0; font-size: 11px; color: #b2bec3; text-transform: uppercase; letter-spacing: 0.5px;'>Comparar precios en:</p>
                    <div style='display: flex; flex-wrap: wrap; gap: 6px;'>
                        <a href='{l_ikea}' target='_blank' style='background: #0058a3; color: white; padding: 6px 12px; border-radius: 4px; font-size: 11px; text-decoration: none; font-weight: 600;'>IKEA</a>
                        <a href='{l_amazon}' target='_blank' style='background: #FF9900; color: #111; padding: 6px 12px; border-radius: 4px; font-size: 11px; text-decoration: none; font-weight: 600;'>Amazon</a>
                        <a href='{l_leroy}' target='_blank' style='background: #73c322; color: white; padding: 6px 12px; border-radius: 4px; font-size: 11px; text-decoration: none; font-weight: 600;'>Leroy Merlin</a>
                        <a href='{l_zara}' target='_blank' style='background: #111; color: white; padding: 6px 12px; border-radius: 4px; font-size: 11px; text-decoration: none; font-weight: 600;'>Zara Home</a>
                    </div>
                </div>
                """
    html_links += "</div></div>"

    # Estructura y generación
    print("📐 Extrayendo plano de la habitación (Canny Edge)...")
    imagen_cv = np.array(imagen_original)
    bordes = cv2.Canny(imagen_cv, 100, 200)
    imagen_bordes = Image.fromarray(np.stack([bordes, bordes, bordes], axis=2))

    print("🎨 Dibujando 3 opciones del nuevo diseño...")
    prompt_negativo = "low quality, bad anatomy, worst quality, cartoon, illustration, distorted, messy"

    imagenes_generadas = pipe(
        prompt_sd,
        negative_prompt=prompt_negativo,
        image=imagen_bordes,
        num_inference_steps=25,
        num_images_per_prompt=3
    ).images

    return analisis_cliente, html_links, prompt_sd, imagen_original, imagen_bordes, imagenes_generadas

def obtener_3_estilos(ruta_imagen, peticion):
    # Cargamos la imagen rápidamente para que el Director la vea
    if ruta_imagen.startswith('http'):
        response = requests.get(ruta_imagen)
        img = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        img = Image.open(ruta_imagen).convert("RGB")

    print("👔 Director de Agencia: Analizando la foto para proponer 3 vías de diseño...")

    prompt = f"""
    Eres el Director de una Agencia de Diseño de Interiores. El cliente ha subido una foto de su habitación y su petición es: '{peticion}'.

    REGLAS ESTRICTAS:
    1. Si el cliente ha pedido un estilo concreto (ej: "quiero estilo nórdico"), el Estilo 1 será su petición mejorada, y los Estilos 2 y 3 serán alternativas tuyas que encajen bien con la arquitectura de la foto.
    2. Si el cliente dice "no lo sé", "sorpréndeme", o no indica nada claro, propón tú los 3 mejores estilos distintos para esa estancia.

    Devuelve EXACTAMENTE 3 líneas de texto. Nada de introducciones, ni viñetas, ni texto extra. Solo el nombre y una breve descripción del estilo en cada línea.

    EJEMPLO DE RESPUESTA:
    Estilo Industrial con toques de madera oscura y metal
    Estilo Nórdico Minimalista con tonos blancos y mucha luz natural
    Estilo Bohemio Cálido con plantas y alfombras étnicas
    """

    respuesta = gemini_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[img, prompt]
    )

    # Limpiamos la respuesta para asegurarnos de que tenemos una lista de 3 textos
    estilos = [linea.strip() for linea in respuesta.text.strip().split('\n') if linea.strip() and len(linea) > 5]

    # Seguro antierrores por si la IA se confunde
    if len(estilos) < 3:
        estilos.extend(["Estilo Moderno Elegante", "Estilo Rústico Acogedor", "Estilo Minimalista Contemporáneo"])

    return estilos[:3]



## Creación de la Interfaz Web (Gradio)
Para que nuestro modelo de IA sea un producto comercializable (SaaS), empaquetamos toda la lógica anterior en una aplicación web interactiva utilizando `Gradio`.

Esta celda construye el *Front-end*: genera un panel para subir imágenes, captura las peticiones del usuario, e instancia un diseño de 3 pestañas dinámicas donde se muestran los análisis, la galería de imágenes generadas y los catálogos de compra con botones HTML. Al ejecutarse, nos dará un enlace público para acceder a la plataforma.

In [ ]:
import gradio as gr
import urllib

# INTERFAZ WEB
def ejecutar_proyecto_web(imagen_ruta, peticion):
    # 1. Filtro de "Cliente Indeciso"
    if not peticion or peticion.strip() == "":
        peticion = "No lo sé, no tengo ni idea de decoración. Sorpréndeme con los 3 estilos que creas que le quedan mejor a esta arquitectura."

    # 2. El Director decide los 3 estilos
    tres_estilos = obtener_3_estilos(imagen_ruta, peticion)

    paquete_resultados = []

    # 3. El Motor procesa los 3 estilos en bucle
    for estilo in tres_estilos:
        analisis, lista_html, prompt_usado, img_orig, img_bordes, opciones_generadas = transformar_habitacion(imagen_ruta, estilo)

        # Preparamos los textos y la galería para la web
        titulo_pestaña = f"## ✨ {estilo.upper()}"
        texto_analisis = f"**Análisis de nuestro experto:**\n\n{analisis}"
        galeria_fotos = [img_orig] + opciones_generadas # Ponemos la original la primera para comparar

        # Guardamos los 4 elementos de esta propuesta
        paquete_resultados.extend([titulo_pestaña, texto_analisis, galeria_fotos, lista_html])

    return paquete_resultados

# CONSTRUCCIÓN DE LA PÁGINA WEB
with gr.Blocks(theme=gr.themes.Monochrome(), title="DECO.AI Premium") as app:
    gr.HTML("""
    <div style='text-align: center; padding: 20px;'>
        <h1 style='color: #2c3e50; font-size: 3em; margin-bottom: 0;'>🛋️ DECO.AI Premium</h1>
        <p style='color: #7f8c8d; font-size: 1.2em;'>Tu estudio de interiorismo con Inteligencia Artificial.</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            # Input de la imagen
            in_img = gr.Image(label="1. Sube la foto de tu habitación", type="filepath")
            in_prompt = gr.Textbox(
                label="2. ¿Tienes algún estilo en mente?",
                placeholder="Ej: Estilo rústico con madera... (Si no lo sabes, déjalo en blanco y te sorprenderemos)"
            )
            btn = gr.Button("🪄 Generar Mi Proyecto Integral (Tardará ~1 min)", variant="primary", size="lg")

        with gr.Column(scale=2):
            # Creamos 3 pestañas dinámicas para los resultados
            componentes_salida = []

            with gr.Tabs():
                for i in range(3):
                    with gr.Tab(f"Propuesta {i+1}"):
                        out_titulo = gr.Markdown(f"### Esperando imagen para la Propuesta {i+1}...")
                        out_analisis = gr.Markdown()
                        # Galería adaptada para mostrar 4 fotos de golpe (1 Original + 3 Opciones)
                        out_galeria = gr.Gallery(label="Desliza para ver las alternativas", columns=4, height="auto")
                        out_catalogo = gr.HTML()

                        componentes_salida.extend([out_titulo, out_analisis, out_galeria, out_catalogo])

    # Conectamos el botón con la función
    btn.click(
        fn=ejecutar_proyecto_web,
        inputs=[in_img, in_prompt],
        outputs=componentes_salida
    )

# Lanzamos el servidor web
app.launch(share=True, debug=True)